# Data Playground: Annotation Analysis
Analysis of custom annotations and MDJCUE annotations with entry/exit point counts

In [2]:
import json
import os
from pathlib import Path
import pandas as pd
from collections import defaultdict

## Load Annotation Data

In [3]:
# Define paths
custom_annotations_dir = "../../data/custom/annotations"
mdjcue_annotations_dir = "../../data/M-DJCUE/annotations"

def load_annotations_from_directory(directory):
    """
    Load all JAMS files from a directory and extract entry/exit point information.
    Returns a dictionary with stats for each annotation file.
    """
    annotations = {}
    
    if not os.path.exists(directory):
        print(f"Directory not found: {directory}")
        return annotations
    
    jams_files = list(Path(directory).glob("*.jams"))
    print(f"Found {len(jams_files)} JAMS files in {directory}")
    
    for jams_file in jams_files:
        try:
            with open(jams_file, 'r') as f:
                data = json.load(f)
            
            # Extract cue_point annotations
            in_count = 0
            out_count = 0
            
            for annotation in data.get('annotations', []):
                if annotation.get('namespace') == 'cue_point':
                    for point in annotation.get('data', []):
                        time = point.get('time', 0)
                        label = point.get('value', {}).get('label', '')
                        
                        # Count nonzero time entries
                        if time > 0:
                            if label == 'IN':
                                in_count += 1
                            elif label == 'OUT':
                                out_count += 1
            
            annotations[jams_file.name] = {
                'path': str(jams_file),
                'entry_points': in_count,
                'exit_points': out_count,
                'total_points': in_count + out_count
            }
        
        except Exception as e:
            print(f"Error loading {jams_file.name}: {e}")
    
    return annotations

# Load both datasets
print("Loading annotations...\n")
custom_annotations = load_annotations_from_directory(custom_annotations_dir)
print()
mdjcue_annotations = load_annotations_from_directory(mdjcue_annotations_dir)

Loading annotations...

Found 109 JAMS files in ../../data/custom/annotations

Found 132 JAMS files in ../../data/M-DJCUE/annotations


## Custom Annotations Summary

In [4]:
if custom_annotations:
    custom_df = pd.DataFrame.from_dict(custom_annotations, orient='index')
    print(f"\n{'='*60}")
    print("CUSTOM ANNOTATIONS SUMMARY")
    print(f"{'='*60}")
    print(f"Total Files: {len(custom_df)}")
    print(f"\nEntry Points (IN) with time > 0:")
    print(f"  Total: {custom_df['entry_points'].sum()}")
    print(f"  Average per file: {custom_df['entry_points'].mean():.2f}")
    print(f"  Max: {custom_df['entry_points'].max()}")
    print(f"  Files with entry points: {(custom_df['entry_points'] > 0).sum()}")
    
    print(f"\nExit Points (OUT) with time > 0:")
    print(f"  Total: {custom_df['exit_points'].sum()}")
    print(f"  Average per file: {custom_df['exit_points'].mean():.2f}")
    print(f"  Max: {custom_df['exit_points'].max()}")
    print(f"  Files with exit points: {(custom_df['exit_points'] > 0).sum()}")
    
    print(f"\nTotal Cue Points: {custom_df['total_points'].sum()}")
    print(f"Files with no annotations: {(custom_df['total_points'] == 0).sum()}")
    print(f"{'='*60}")
    
    # Show detailed breakdown
    print("\nDetailed Breakdown (sorted by total points):")
    custom_df_sorted = custom_df.sort_values('total_points', ascending=False)
    display(custom_df_sorted[['entry_points', 'exit_points', 'total_points']])
else:
    print("No custom annotations found.")


CUSTOM ANNOTATIONS SUMMARY
Total Files: 109

Entry Points (IN) with time > 0:
  Total: 244
  Average per file: 2.24
  Max: 3
  Files with entry points: 109

Exit Points (OUT) with time > 0:
  Total: 230
  Average per file: 2.11
  Max: 3
  Files with exit points: 109

Total Cue Points: 474
Files with no annotations: 0

Detailed Breakdown (sorted by total points):


,entry_points,exit_points,total_points
CamelPhat & ARTBAT - For a Feeling (feat. RHODES).jams,3,3,6
ARTBAT - Closer (feat. WhoMadeWho).jams,3,3,6
Echonomist & Jenia Tarsol - Happiest of all memorial days (feat. Acollective).jams,3,3,6
Moderat - More Love (Rampa &ME Remix).jams,3,3,6
Mind Against - Walking Away.jams,3,3,6
...,...,...,...
MEDUZA & James Carter - Bad Memories (feat. Elley Duhé & FAST BOY).jams,1,2,3
"Adam Port, Stryv & Malachiii - Move.jams",1,2,3
Black Coffee - Crazy (feat. Thiwe).jams,1,1,2
ARTBAT & Sailor & I - Best of Me.jams,1,1,2


## MDJCUE Annotations Summary

In [6]:
if mdjcue_annotations:
    mdjcue_df = pd.DataFrame.from_dict(mdjcue_annotations, orient='index')
    print(f"\n{'='*60}")
    print("MDJCUE ANNOTATIONS SUMMARY")
    print(f"{'='*60}")
    print(f"Total Files: {len(mdjcue_df)}")
    print(f"\nEntry Points (IN) with time > 0:")
    print(f"  Total: {mdjcue_df['entry_points'].sum()}")
    print(f"  Average per file: {mdjcue_df['entry_points'].mean():.2f}")
    print(f"  Max: {mdjcue_df['entry_points'].max()}")
    print(f"  Files with entry points: {(mdjcue_df['entry_points'] > 0).sum()}")
    
    print(f"\nExit Points (OUT) with time > 0:")
    print(f"  Total: {mdjcue_df['exit_points'].sum()}")
    print(f"  Average per file: {mdjcue_df['exit_points'].mean():.2f}")
    print(f"  Max: {mdjcue_df['exit_points'].max()}")
    print(f"  Files with exit points: {(mdjcue_df['exit_points'] > 0).sum()}")
    
    print(f"\nTotal Cue Points: {mdjcue_df['total_points'].sum()}")
    print(f"Files with no annotations: {(mdjcue_df['total_points'] == 0).sum()}")
    print(f"{'='*60}")
    
    # Show detailed breakdown
    print("\nDetailed Breakdown (sorted by total points):")
    mdjcue_df_sorted = mdjcue_df.sort_values('total_points', ascending=False)
    display(mdjcue_df_sorted[['entry_points', 'exit_points', 'total_points']].head(20))
else:
    print("No MDJCUE annotations found.")


MDJCUE ANNOTATIONS SUMMARY
Total Files: 132

Entry Points (IN) with time > 0:
  Total: 997
  Average per file: 7.55
  Max: 15
  Files with entry points: 132

Exit Points (OUT) with time > 0:
  Total: 797
  Average per file: 6.04
  Max: 12
  Files with exit points: 132

Total Cue Points: 1794
Files with no annotations: 0

Detailed Breakdown (sorted by total points):


,entry_points,exit_points,total_points
v_MCJ ft Davina - I'm Ready (For Your Love) (The Get Ready mix).jams,15,7,22
Jose Nunez - Bilingual (Dirty Mix).jams,11,10,21
v_Fonda Rae - Living In Ecstasy (I Like What You Do) (The Groove Mix).jams,13,8,21
Ron Trent - Altered States.jams,11,10,21
Harry Romero ft Robert Owens - I Go Back.jams,10,10,20
v_C.J. Bolland - Sugar Is Sweeter (Armand's Drum 'n' Bass Mix).jams,10,9,19
v_Kariya - Let Me Love You For Tonight (Original House Mix).jams,12,7,19
v_Barbara Tucker - Stay Together (Armand's Crazy Trauma Mix).jams,11,8,19
v_Jaydee - Plastic Dreams (Morales Club Mix).jams,7,11,18
v_Urban Blues Project pres. Michael Procter - Love Don't Live (New Birth (Mix) UK Re-Edit).jams,10,8,18


In [7]:
if custom_annotations and mdjcue_annotations:
    print(f"\n{'='*60}")
    print("COMPARATIVE SUMMARY")
    print(f"{'='*60}")
    print(f"{'Metric':<30} {'Custom':<15} {'MDJCUE':<15}")
    print("-" * 60)
    print(f"{'Total Files':<30} {len(custom_df):<15} {len(mdjcue_df):<15}")
    print(f"{'Total Entry Points':<30} {custom_df['entry_points'].sum():<15} {mdjcue_df['entry_points'].sum():<15}")
    print(f"{'Total Exit Points':<30} {custom_df['exit_points'].sum():<15} {mdjcue_df['exit_points'].sum():<15}")
    print(f"{'Total Cue Points':<30} {custom_df['total_points'].sum():<15} {mdjcue_df['total_points'].sum():<15}")
    print(f"{'Avg Entry Points/File':<30} {custom_df['entry_points'].mean():<15.2f} {mdjcue_df['entry_points'].mean():<15.2f}")
    print(f"{'Avg Exit Points/File':<30} {custom_df['exit_points'].mean():<15.2f} {mdjcue_df['exit_points'].mean():<15.2f}")
    print(f"{'Files with Annotations':<30} {(custom_df['total_points'] > 0).sum():<15} {(mdjcue_df['total_points'] > 0).sum():<15}")
    print(f"{'% Annotated':<30} {(custom_df['total_points'] > 0).sum()/len(custom_df)*100:<14.1f}% {(mdjcue_df['total_points'] > 0).sum()/len(mdjcue_df)*100:<14.1f}%")
    print(f"{'='*60}")


COMPARATIVE SUMMARY
Metric                         Custom          MDJCUE         
------------------------------------------------------------
Total Files                    109             132            
Total Entry Points             244             997            
Total Exit Points              230             797            
Total Cue Points               474             1794           
Avg Entry Points/File          2.24            7.55           
Avg Exit Points/File           2.11            6.04           
Files with Annotations         109             132            
% Annotated                    100.0         % 100.0         %
